# 06 – Cross-Validation

## Introduction

This notebook focuses on evaluating the consistency and generalisation performance of four machine learning regression models developed to predict menstrual cycle length.

In the previous model evaluation, the models were assessed using a single train-test split. However, since the dataset contains only 81 records, the performance obtained from a single split may not provide a reliable estimate of model performance.

To address this limitation, 5-fold cross-validation is performed. The dataset is divided into five subsets, where each subset is used once for validation while the remaining four subsets are used for training.

The following regression models are evaluated:

1. Linear Regression
2. Random Forest Regressor
3. Gradient Boosting Regressor
4. Support Vector Regression (SVR)

Model performance is assessed using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² score. The mean and standard deviation of these metrics are calculated across the five folds.

A baseline regression model is also evaluated to determine whether the machine learning models provide better predictions than simply using the average cycle length.

The main objective is to identify the most promising model based on prediction accuracy and consistency across different data splits.

In [2]:
# Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.dummy import DummyRegressor

In [3]:
# Load cleaned dataset

df = pd.read_csv("../Dataset/menstrual_health_cleaned.csv")

print("Dataset shape:", df.shape)

df.head()

# Separate features and target

X = df.drop(columns=[
    "Length_of_cycle",
    "Mean_of_length_of_cycle",
    "Estimated_day_of_ovulution",
    "Weight"
])

y = df["Length_of_cycle"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

# Identify feature types

categorical_features = ["Unusual_Bleeding"]

numerical_features = [
    "number_of_peak",
    "Age",
    "Length_of_Leutal_Phase",
    "Length_of_menses",
    "BMI",
    "Menses_score"
]

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

# Preprocessing for numerical features

numerical_transformer = StandardScaler()

# Preprocessing for categorical features

categorical_transformer = OneHotEncoder(
    drop="if_binary",
    handle_unknown="ignore"
)

# Combine preprocessing steps

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")


# Define regression models

models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    ),

    "Support Vector Regression": SVR(
        kernel="rbf",
        C=1.0,
        epsilon=0.1
    )
}

print("Four regression models defined successfully!")





Dataset shape: (81, 11)
Feature shape: (81, 7)
Target shape: (81,)
Numerical features: ['number_of_peak', 'Age', 'Length_of_Leutal_Phase', 'Length_of_menses', 'BMI', 'Menses_score']
Categorical features: ['Unusual_Bleeding']
Preprocessing pipeline created successfully!
Four regression models defined successfully!


In [3]:
# Define 5-fold cross-validation

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold cross-validation configured successfully!")

5-fold cross-validation configured successfully!


In [4]:
# Evaluate all regression models

results = []

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=kf,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        }
    )

    results.append({
        "Model": name,
        "Mean MAE": -scores["test_MAE"].mean(),
        "SD MAE": scores["test_MAE"].std(),
        "Mean RMSE": -scores["test_RMSE"].mean(),
        "SD RMSE": scores["test_RMSE"].std(),
        "Mean R2": scores["test_R2"].mean(),
        "SD R2": scores["test_R2"].std()
    })

print("Cross-validation completed successfully!")

Cross-validation completed successfully!


In [5]:
# Create results table

cv_results = pd.DataFrame(results)

print(cv_results.round(3))

                       Model  Mean MAE  SD MAE  Mean RMSE  SD RMSE  Mean R2  \
0          Linear Regression     3.151   0.544      4.066    0.738   -0.006   
1              Random Forest     3.532   0.461      4.368    0.669   -0.211   
2          Gradient Boosting     3.535   0.498      4.466    0.619   -0.262   
3  Support Vector Regression     2.987   0.450      4.090    0.590   -0.012   

   SD R2  
0  0.204  
1  0.417  
2  0.410  
3  0.077  


In [6]:

# Define baseline model

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

baseline_scores = cross_validate(
    baseline_pipeline,
    X,
    y,
    cv=kf,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    }
)

print("Baseline Mean MAE:", -baseline_scores["test_MAE"].mean())
print("Baseline Mean RMSE:", -baseline_scores["test_RMSE"].mean())
print("Baseline Mean R2:", baseline_scores["test_R2"].mean())

Baseline Mean MAE: 3.0711962669683257
Baseline Mean RMSE: 4.1808546395482
Baseline Mean R2: -0.0523791548976484


## Interpretation of Results

Five-fold cross-validation was performed to compare the performance of four regression models.

### Model Performance Comparison

Among the four models, Support Vector Regression (SVR) achieved the lowest Mean MAE of 2.987 days, indicating the lowest average absolute prediction error among the evaluated machine learning models.

Linear Regression achieved a Mean MAE of 3.151 days, while Random Forest and Gradient Boosting showed higher errors of 3.532 and 3.535 days, respectively.

### Model Consistency

The standard deviation of MAE indicates how much the prediction error varies across the five folds.

SVR showed an MAE standard deviation of 0.450, suggesting relatively consistent prediction errors across the folds. It also achieved the lowest standard deviation in R² (0.077).

Linear Regression showed a Mean R² score close to zero (-0.006), while SVR achieved -0.012. Both models performed better than Random Forest and Gradient Boosting in terms of mean R².

### Baseline Comparison

The baseline model achieved a Mean MAE of 3.071 days and a Mean RMSE of 4.181 days.

SVR achieved a slightly lower Mean MAE of 2.987 days and Mean RMSE of 4.090 days compared with the baseline.

However, the SVR Mean R² score (-0.012) was slightly worse than the baseline R² score (-0.052).

This indicates that SVR provides a small improvement in prediction error, but its overall explanatory performance remains limited.

### Overall Observation

SVR demonstrated the lowest prediction error among the four machine learning models and relatively consistent performance across the five folds.

However, all four models produced negative mean R² scores, indicating that their predictive performance remains limited.

The small dataset size and potentially limited predictive information in the selected features may contribute to these results.

## Conclusion

Five-fold cross-validation was successfully performed to assess the consistency and generalisation performance of four machine learning regression models for menstrual cycle length prediction.

Among the evaluated models, Support Vector Regression (SVR) achieved the lowest Mean MAE of 2.987 days and a Mean RMSE of 4.090 days. It also demonstrated relatively low variation in prediction performance across the five folds.

Compared with the baseline model, SVR showed a small improvement in MAE and RMSE. However, its Mean R² score remained negative, indicating that the current model has limited ability to explain variations in menstrual cycle length.

These findings suggest that the selected features and available dataset may not provide sufficient predictive information for accurate cycle-length prediction.

Since the dataset contains only 81 records, further investigation using a larger and more diverse dataset is necessary to improve model reliability and generalisation.

The results of this evaluation provide a basis for selecting a model for further investigation using Explainable AI techniques such as SHAP, while keeping its predictive limitations in consideration.